In [1]:
from dotenv import load_dotenv
from datetime import datetime, timedelta

import os
import json
import openai
import requests

# Load environment variables from .env file
load_dotenv('~/Documents/Jupyter/MDST/WN25/Eventure/EventureApp/.env')

openai.api_key = os.getenv("OPENAI_API_KEY")
ticketmaster_key = os.getenv("TICKETMASTER_API_KEY")

# Verify
print("OpenAI Key Found:", bool(os.getenv("OPENAI_API_KEY")))

MODEL_NAME = "gpt-4o-mini"
client = openai.OpenAI()


OpenAI Key Found: True


Create Functions:

In [2]:
# def makeTicketmasterEventRequest(eventparams: str):
#     # Format for looking at events: https://app.ticketmaster.com/{package}/{version}/events.json?{eventparams}apikey={API key}
#     # eventparams needs to be separated by & symbols for each query. Check ticketmaster discovery page to see all query options

#     url = f'https://app.ticketmaster.com/discovery/v2/events.json?{eventparams}&apikey={ticketmaster_key}'

#     r = requests.get(url)

#     return r 

def makeTicketmasterEventRequest(params: dict):
    print(f"--- Calling Ticketmaster API with params: {params} ---")
    params['apikey'] = ticketmaster_key
    url = f'https://app.ticketmaster.com/discovery/v2/events.json'

    r = requests.get(url,params=params)
    print(f"--- Ticketmaster API Status Code: {r.status_code} ---")

    return r.text

In [3]:
def makeTicketmasterVenuesRequest(params: dict):
    print(f"--- Calling Ticketmaster API with params: {params} ---")
    params['apikey'] = ticketmaster_key
    url = f'https://app.ticketmaster.com/discovery/v2/venues.json'

    r = requests.get(url,params=params)
    print(f"--- Ticketmaster API Status Code: {r.status_code} ---")

    return r.text

Describe functions to OpenAI:

In [4]:
function_descriptions = [
    {
        "type": "function",
        "function": {
            "name": "makeTicketmasterEventRequest",
            "description": "Fetches all events happening in a specific location from Ticketmaster.",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "Search term for events, e.g., 'music', 'sports'."
                    },
                    "city": {
                        "type": "string",
                        "description": "The city to fetch events for, e.g., 'Detroit'."
                    },
                    "stateCode": {
                        "type": "string",
                        "description": "The state code, e.g., 'MI' for Michigan."
                    },
                    "countryCode": {
                        "type": "string",
                        "description": "The country code, e.g., 'US'."
                    },
                    "startDateTime": {
                        "type": "string",
                        "format": "date-time",
                        "description": "Start date and time for events in ISO 8601 format (YYYY-MM-DDTHH:MM:SSZ)."
                    },
                    "endDateTime": {
                        "type": "string",
                        "format": "date-time",
                        "description": "End date and time for events in ISO 8601 format."
                    }
                },
                "required": ["city", "stateCode", "countryCode"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "makeTicketmasterVenuesRequest",
            "description": "Get details for a specific venue using the unique identifier for the venue.",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "Search term for events, e.g., 'music', 'sports'."
                    },
                    "stateCode": {
                        "type": "string",
                        "description": "The state code, e.g., 'MI' for Michigan."
                    },
                    "countryCode": {
                        "type": "string",
                        "description": "The country code, e.g., 'US'."
                    },
                },
                "required": ["stateCode", "countryCode"]
            }
        }
    }
]


In [5]:
def test_call_model(user_message: str, function_call="auto"):
    """
    1) We send user_message + function_descriptions to the model
    2) We see if it returns a function call
    3) If so, we parse arguments, call the function ourselves
    4) Return final output
    """

    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": user_message}],
        tools=function_descriptions,  # 'functions' is now called 'tools'
        tool_choice=function_call,    # 'function_call' is now 'tool_choice'
    )
    response = completion.choices[0].message
    return response

In [6]:
def run_conversation(user_prompt_1):
  messages = []
  messages.append({"role": "user", "content": user_prompt_1})

  response_message = test_call_model(user_prompt_1)
  messages.append(response_message.model_dump(exclude_unset=True))

  tool_calls = response_message.tool_calls
  
  available_functions = {
    "makeTicketmasterEventRequest": makeTicketmasterEventRequest,
    "makeTicketmasterVenuesRequest": makeTicketmasterVenuesRequest
    # we can add more functions here <--
  }

  tool_responses = []

  for tool_call in tool_calls:
    function_name = tool_call.function.name
    tool_call_id = tool_call.id

    function_to_call = available_functions[function_name]

    function_args = json.loads(tool_call.function.arguments)
    function_response_data = function_to_call(params=function_args)

    tool_responses.append(
        {
          "tool_call_id": tool_call_id,
          "role": "tool",
          "name": function_name,
          "content": function_response_data,
      }
    )

  messages.extend(tool_responses)

  second_response = client.chat.completions.create(
      model=MODEL_NAME,
      messages=messages,
  )
  final_response_message = second_response.choices[0].message
  messages.append(final_response_message.model_dump(exclude_unset=True))
  final_response_content = final_response_message.content

  return final_response_content, messages

In [9]:
from datetime import date, timedelta
today = date.today()
start_date = today.strftime("%Y-%m-%dT00:00:00Z")
end_date = (today + timedelta(days=7)).strftime("%Y-%m-%dT23:59:59Z")

answer, current_history = run_conversation(
    f"What venues are available in Michigan?",
)
print(f"Answer: {answer}")

answer, current_history = run_conversation(
    f"What events are going on today in Detroit?",
)
print(f"Answer: {answer}")

--- Calling Ticketmaster API with params: {'stateCode': 'MI', 'countryCode': 'US'} ---
--- Ticketmaster API Status Code: 200 ---
Answer: Here are some popular venues available in Michigan:

1. **[Little Caesars Arena](https://www.ticketmaster.com/little-caesars-arena-tickets-detroit/venue/66238)**  
   - **Location**: 2645 Woodward, Detroit, MI 48201  
   - **Capacity**: 20,000  
   - **Box Office Phone**: (313) 471-7929  

   ![Little Caesars Arena](https://s1.ticketm.net/dbimages/23263v.jpg)

2. **[Ford Field](https://www.ticketmaster.com/ford-field-tickets-detroit/venue/65869)**  
   - **Location**: 2000 Brush St., Detroit, MI 48226  
   - **Capacity**: 65,000  
   - **Box Office Phone**: (313) 262-2000  

   ![Ford Field](https://s1.ticketm.net/dbimages/FordField_MI_65869_4Z.gif)

3. **[Fox Theatre Detroit](https://www.ticketmaster.com/fox-theatre-detroit-tickets-detroit/venue/65547)**  
   - **Location**: 2211 Woodward Ave, Detroit, MI 48201  
   - **Capacity**: 5,048  
   - **Box

In [8]:
# Let's do a quick test with something that calls 'get_weather_info'
user_prompt_1 = "What events are going on in Michigan for this month"
resp = test_call_model(user_prompt_1, function_call="auto")
print("Model response:\n", resp)


Model response:
 ChatCompletionMessage(content=None, refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_l6OuRNLXx1H0YG3GDCNTz5zM', function=Function(arguments='{"city":"Detroit","stateCode":"MI","countryCode":"US"}', name='makeTicketmasterEventRequest'), type='function')], annotations=[])


## **TODO:** Fix the OpenAI Instance 
- Fix the 'function_call=None' to be 'function_call=makeTicketmasterEventRequest'
- Assumed because 'function_descriptions' is wrong, or 'makeTicketmasterEventRequest' is broken